# Laboratorio 2 — Calidad y preparación de tu conjunto de datos
**Ciencia de Datos · Semana 2 · Advance UCSC**

| | |
|---|---|
| **Resultado de aprendizaje** | RA1 (cierre) · RA3 (transversal) |
| **Duración estimada** | 1 h asincrónica + 1 h de trabajo autónomo |
| **Insumo** | El conjunto de datos que elegiste en la semana 1 |
| **Producto** | Conjunto limpio + bitácora completada + avance versionado |
| **Estudiante** | _(escribe tu nombre aquí)_ |

---

Este laboratorio **no trae los datos**: los traes tú. Todo lo que hagas aquí es el cuerpo de la **Entrega 1**, que rindes al cierre de esta semana.

Ten abierta en paralelo la planilla `S2_Bitacora_Decisiones_CDD.xlsx`. Escríbela **mientras trabajas**, no al final: reconstruir después por qué tomaste una decisión es mucho más difícil de lo que parece.

In [32]:
import pandas as pd
import numpy as np

RUTA = ''
SEMILLA = 20262
np.random.seed(SEMILLA)

ARCHIVO = 'winequality-red.csv'   # <- reemplaza
SEPARADOR = ';'                        # <- ajusta si corresponde

df = pd.read_csv(RUTA + ARCHIVO, sep=SEPARADOR)
print('Forma:', df.shape)
df.head()

Forma: (1599, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [33]:
df.dtypes

,0
fixed acidity,float64
volatile acidity,float64
citric acid,float64
residual sugar,float64
chlorides,float64
free sulfur dioxide,float64
total sulfur dioxide,float64
density,float64
pH,float64
sulphates,float64


## Parte 1 · Diagnóstico de las cinco dimensiones

Antes de limpiar nada, hay que saber qué tienes. Completa la hoja **«Resumen de calidad»** de la bitácora con los resultados de esta parte.

### 1.1 Completitud

In [34]:
faltantes = df.isna().sum().to_frame('faltantes')
faltantes['pct'] = (faltantes['faltantes'] / len(df) * 100).round(2)
faltantes.sort_values('faltantes', ascending=False)

,faltantes,pct
fixed acidity,0,0.0
volatile acidity,0,0.0
citric acid,0,0.0
residual sugar,0,0.0
chlorides,0,0.0
free sulfur dioxide,0,0.0
total sulfur dioxide,0,0.0
density,0,0.0
pH,0,0.0
sulphates,0,0.0


### ✏️ Actividad 1.1

Elige la columna con más faltantes y averigua **si se reparten al azar o se concentran en algún grupo**. Usa `pd.crosstab` contra alguna columna categórica, como en el Ejercicio 1 del cuaderno resuelto.

Es la diferencia entre poder imputar y tener que declarar.

In [ ]:
# Tu código aquí


# **No existen datos faltantes en el dataset**

### 1.2 Exactitud — rangos y valores imposibles

In [35]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
fixed acidity,1599.0,8.32,1.74,4.60,7.10,7.90,9.20,15.90
volatile acidity,1599.0,0.53,0.18,0.12,0.39,0.52,0.64,1.58
citric acid,1599.0,0.27,0.19,0.00,0.09,0.26,0.42,1.00
residual sugar,1599.0,2.54,1.41,0.90,1.90,2.20,2.60,15.50
chlorides,1599.0,0.09,0.05,0.01,0.07,0.08,0.09,0.61
free sulfur dioxide,1599.0,15.87,10.46,1.00,7.00,14.00,21.00,72.00
total sulfur dioxide,1599.0,46.47,32.90,6.00,22.00,38.00,62.00,289.00
density,1599.0,1.00,0.00,0.99,1.00,1.00,1.00,1.00
pH,1599.0,3.31,0.15,2.74,3.21,3.31,3.40,4.01
sulphates,1599.0,0.66,0.17,0.33,0.55,0.62,0.73,2.00


### ✏️ Actividad 1.2

Mira los mínimos y máximos. **¿Hay valores imposibles para lo que la columna representa?** Busca en particular valores centinela: `-1`, `0`, `999`, `9999`, `-999`, o cadenas como `'S/I'`, `'NULL'`, `'sin dato'`.

Recuerda: un centinela numérico **no** se delata en los tipos de dato.

In [43]:

for col in df.select_dtypes(include='number').columns:
    print(f"{col:20s} Min: {df[col].min():<8} Max: {df[col].max():<8}")
print("\n")

centinelas = [-1, 0, 999, 9999, -999]

for columna in df.columns:
    for valor in centinelas:
        cantidad = (df[columna] == valor).sum()

        if cantidad > 0:
            print(
                f"Columna: {columna} | "
                f"Valor: {valor} | "
                f"Cantidad: {cantidad} \n"
            )

centinelas_texto = ['S/I', 'NULL', 'sin dato']

for columna in df.columns:
    valores = df[columna].astype('string').str.strip().str.upper()

    for valor in centinelas_texto:
        cantidad = (valores == valor.upper()).sum()

        if cantidad > 0:
            print(
                f"Columna: {columna} | "
                f"Valor: {valor} | "
                f"Cantidad: {cantidad}\n"
            )



fixed acidity        Min: 4.6      Max: 15.9    
volatile acidity     Min: 0.12     Max: 1.58    
citric acid          Min: 0.0      Max: 1.0     
residual sugar       Min: 0.9      Max: 15.5    
chlorides            Min: 0.012    Max: 0.611   
free sulfur dioxide  Min: 1.0      Max: 72.0    
total sulfur dioxide Min: 6.0      Max: 289.0   
density              Min: 0.99007  Max: 1.00369 
pH                   Min: 2.74     Max: 4.01    
sulphates            Min: 0.33     Max: 2.0     
alcohol              Min: 8.4      Max: 14.9    
quality              Min: 3        Max: 8       


Columna: citric acid | Valor: 0 | Cantidad: 132 



### 1.3 Consistencia — reglas cruzadas

### ✏️ Actividad 1.3

Escribe **dos reglas de consistencia** que tu conjunto debería cumplir y verifícalas. Ejemplos:

- «La fecha de término no puede ser anterior a la de inicio.»
- «Si el estado es *cerrado*, el monto vigente debe estar vacío.»
- «La suma de las partes debe igualar el total.»

Estas reglas no salen de los datos: salen de entender qué representan.

In [37]:
#Reglas de negocio planteadas:

# Regla 1: SO2 Libre <= SO2 Total
'''En la química del vino, el dióxido de azufre total es la suma de dos partes: el SO₂
que está libre (activo como conservante) más el SO₂ que ya se combinó con otras moléculas del vino.
Por lo tanto, es físicamente imposible que la parte libre supere al total'''

inconsistentes_so2 = df[df['free sulfur dioxide'] > df['total sulfur dioxide']]
print("Violaciones Regla 1 (SO2):", len(inconsistentes_so2))



# Regla 2: Rangos válidos de pH y Alcohol

'''Un vino tinto por definición es una bebida ácida (el pH neutro es 7; un vino tinto real suele fluctuar entre 3.0 y 4.0).
Un pH inferior a 2.5 sería demasiado corrosivo y superior a 4.5 indicaría un vino estropeado o una medición errónea.
Del mismo modo, un vino no puede tener $0\%$ o valores negativos de alcohol.'''

inconsistentes_ph_alc = df[(df['pH'] < 2.5) | (df['pH'] > 4.5) | (df['alcohol'] <= 0)]
print("Violaciones Regla 2 (pH/Alcohol):", len(inconsistentes_ph_alc))


Violaciones Regla 1 (SO2): 0
Violaciones Regla 2 (pH/Alcohol): 0


### 1.4 Unicidad

In [38]:
print('Duplicados exactos:', df.duplicated().sum())
# Si tu conjunto tiene una columna identificadora, revísala también:
# print('ID repetidos:', df['TU_ID'].duplicated().sum())

Duplicados exactos: 240


### ✏️ Actividad 1.4

Construye una **clave normalizada** (sin espacios, en mayúsculas, sin tildes) con las columnas que deberían identificar un registro, y cuenta los casi-duplicados.

In [39]:
def normalizar_texto(serie):
    return (serie.astype(str).str.strip().str.upper()
                 .str.normalize('NFKD').str.encode('ascii', 'ignore').str.decode('ascii'))

# Crear clave normalizada con las variables principales para contar duplicados exactos y casi-duplicados
df['clave_norm'] = (
    normalizar_texto(df['fixed acidity']) + '_' +
    normalizar_texto(df['volatile acidity']) + '_' +
    normalizar_texto(df['citric acid']) + '_' +
    normalizar_texto(df['alcohol'])
)

print('Duplicados exactos:', df.duplicated().sum())
print('Duplicados por clave normalizada:', df['clave_norm'].duplicated().sum())
df.drop(columns=['clave_norm'], inplace=True)


Duplicados exactos: 240
Duplicados por clave normalizada: 246


### 1.5 Oportunidad

In [ ]:
# Si tienes alguna columna de fecha, ¿cuál es el registro más reciente?
# ¿El conjunto está vigente para la pregunta que quieres respond


**Detalle Oportunidad**: El conjunto proviene del estudio clásico de Cortez et al. (2009). Al no poseer estampas de tiempo (timestamps), se declara en la bitácora como limitación declarada por ser una muestra estática

---
## Parte 2 · Limpieza, decisión por decisión

Ahora corriges. **Regla del laboratorio: cada celda de esta parte tiene su fila en la bitácora.** Si aplicas una transformación que no registraste, no cuenta.

Recuerda las tres decisiones legítimas: **Corregir**, **Descartar** o **Declarar como limitación**.

### ✏️ Actividad 2.1 — Centinelas a faltantes

In [ ]:
# 2.1 Centinelas: No se detectaron centinelas que requieran conversión a NaN.

### ✏️ Actividad 2.2 — Normalización de categorías

Normaliza mecánicamente y después revisa lo que queda: los plurales, las abreviaturas y los sinónimos necesitan un diccionario de equivalencias que escribes tú.

In [14]:
df.columns = [col.strip().lower().replace(' ', '_') for col in df.columns]


### ✏️ Actividad 2.3 — Duplicados

In [15]:
filas_antes = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Filas eliminadas por duplicidad: {filas_antes - len(df)} | Filas resultantes: {len(df)}")


Filas eliminadas por duplicidad: 240 | Filas resultantes: 1359


### ✏️ Actividad 2.4 — Tipos de dato

Convierte a numérico lo que deba ser numérico y a fecha lo que deba ser fecha. Si hay formatos mixtos, prueba formatos explícitos en orden en vez de dejar que pandas adivine.

In [44]:
for col in df.columns:
    if col != 'quality':
        df[col] = df[col].astype(float)
df['quality'] = df['quality'].astype(int)


### ✏️ Actividad 2.5 — Atípicos

Identifica los valores extremos y **pronúnciate sobre cada grupo**: ¿error de registro o fenómeno real?

Cruza la variable con otra que te permita juzgar plausibilidad, como en el Ejercicio 8. No elimines atípicos solo por serlo.

In [22]:
# Detección de atípicos con el método IQR
for col in df.select_dtypes(include='number').columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    outliers = df[(df[col] < (q1 - 1.5 * iqr)) | (df[col] > (q3 + 1.5 * iqr))]
    print(f"{col:25s} Atípicos: {len(outliers)}")


fixed_acidity             Atípicos: 41
volatile_acidity          Atípicos: 19
citric_acid               Atípicos: 1
residual_sugar            Atípicos: 126
chlorides                 Atípicos: 87
free_sulfur_dioxide       Atípicos: 26
total_sulfur_dioxide      Atípicos: 45
density                   Atípicos: 35
ph                        Atípicos: 28
sulphates                 Atípicos: 55
alcohol                   Atípicos: 12
quality                   Atípicos: 27
total_acidity             Atípicos: 37


_Tu conclusión sobre los atípicos:_


Los valores extremos identificados en variables como total_sulfur_dioxide (hasta 289 mg/dm³) o residual_sugar (hasta 15.5 g/dm³) representan un fenómeno real de elaboración y conservación del vino, no errores de digitalización ni centinelas. Enología e investigación confirman que vinos con mayor azúcar residual o tratamientos específicos de conservación requieren niveles más altos de sulfitos. Se decide conservarlos para no perder variabilidad real que impacte la percepción de calidad.   


---
## Parte 3 · Transformación y nuevos atributos

Preparar no es solo corregir: también es **construir** lo que el análisis va a necesitar.

### ✏️ Actividad 3.1 — Un atributo derivado

Crea **al menos una variable nueva** a partir de las existentes, que aporte a tu pregunta analítica. Ejemplos: una razón entre dos columnas, la antigüedad en años a partir de una fecha, un tramo a partir de un valor continuo.

Explica en la celda siguiente por qué esa variable ayuda a responder tu pregunta.

In [23]:
# Creación de acidez total combinada
df['total_acidity'] = (df['fixed_acidity'] + df['volatile_acidity']).round(4)
df[['fixed_acidity', 'volatile_acidity', 'total_acidity']].head()


,fixed_acidity,volatile_acidity,total_acidity
0,7.4,0.70,8.10
1,7.8,0.88,8.68
2,7.8,0.76,8.56
3,11.2,0.28,11.48
4,7.4,0.66,8.06


_Por qué esta variable aporta a mi pregunta:_

La acidez total (total_acidity) combina la acidez fija (tartárico/málico) y la acidez volátil (acético). Evaluar esta suma permite analizar si la carga ácida global influye en la calidad percibida del vino o si el consumidor detecta negativamente la proporción de acidez volátil respecto al total organoléptico.   


### ✏️ Actividad 3.2 — Normalización o codificación

Si tu conjunto tiene variables numéricas en escalas muy distintas, o categóricas que necesitarás codificar en la semana 4, déjalas preparadas aquí.

In [25]:
from sklearn.preprocessing import StandardScaler

# Seleccionar las variables explicativas continuas con sus nombres originales en inglés
cols_continuas = [
    'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar',
    'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density',
    'ph', 'sulphates', 'alcohol', 'total_acidity'
]

# Crear una copia estandarizada
scaler = StandardScaler()
df_normalizado = df.copy()
df_normalizado[cols_continuas] = scaler.fit_transform(df[cols_continuas])

df_normalizado.head()


,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,ph,sulphates,alcohol,quality,total_acidity
0,-0.524431,0.932000,-1.393258,-0.461157,-0.245623,-0.468554,-0.384050,0.584003,1.291872,-0.578561,-0.954374,5,-0.435617
1,-0.294063,1.915800,-1.393258,0.056665,0.200094,0.872003,0.604073,0.048737,-0.708395,0.124822,-0.584575,5,-0.094221
2,-0.294063,1.259934,-1.188617,-0.165259,0.078535,-0.085537,0.214813,0.155790,-0.321247,-0.051024,-0.584575,5,-0.164855
3,1.664067,-1.363534,1.471711,-0.461157,-0.265883,0.105971,0.394471,0.691057,-0.966495,-0.461331,-0.584575,6,1.553896
4,-0.524431,0.713378,-1.393258,-0.535132,-0.265883,-0.277045,-0.204391,0.584003,1.291872,-0.578561,-0.954374,5,-0.459162


---
## Parte 4 · El flujo como función reutilizable

Junta todo lo anterior en **una sola función**. Esto es lo que evalúa el RA3 y lo que se revisa en la Sección A de la Entrega Final: que tu limpieza se pueda volver a ejecutar de principio a fin.

In [48]:
COLUMNAS_ESPAÑOL = {
    'fixed acidity': 'acidez_fija',
    'volatile acidity': 'acidez_volatil',
    'citric acid': 'acido_citrico',
    'residual sugar': 'azucar_residual',
    'chlorides': 'cloruros',
    'free sulfur dioxide': 'dioxido_azufre_libre',
    'total sulfur dioxide': 'dioxido_azufre_total',
    'density': 'densidad',
    'pH': 'ph',
    'sulphates': 'sulfatos',
    'alcohol': 'alcohol',
    'quality': 'calidad'
}

def limpiar(ruta, sep=SEPARADOR, semilla=SEMILLA):
    """
    Flujo completo de limpieza y preparación del conjunto winequality-red.csv.
    Devuelve: (df_limpio, informe)
    """
    np.random.seed(semilla)

    # 1. Carga inicial
    d = pd.read_csv(ruta, sep=sep)
    informe = {'filas_iniciales': len(d)}

    # 2. Traducción y normalización de encabezados al español (Actividad 2.2)
    d = d.rename(columns=COLUMNAS_ESPAÑOL)

    # 3. Centinelas / Faltantes: Sin nulos ni centinelas numéricos.

    # 4. Duplicados: Descartar duplicados exactos (Actividad 2.3)
    n_duplicados = d.duplicated().sum()
    d = d.drop_duplicates().reset_index(drop=True)
    informe['duplicados_eliminados'] = int(n_duplicados)

    # 5. Tipos de datos (Actividad 2.4)
    for col in d.columns:
        if col != 'calidad':
            d[col] = d[col].astype(float)
    d['calidad'] = d['calidad'].astype(int)

    # 6. Atípicos (Actividad 2.5)
    informe['atipicos_tratamiento'] = 'Conservados (fenómeno real)'

    # 7. Atributo derivado: Acidez total (Actividad 3.1)
    d['acidez_total'] = (d['acidez_fija'] + d['acidez_volatil']).round(4)

    informe['filas_finales'] = len(d)
    return d, informe


# Prueba de ejecución y guardado
limpio, informe = limpiar(RUTA + ARCHIVO)
print("Informe de ejecución:", informe)

# Comprobación de reproducibilidad
a, _ = limpiar(RUTA + ARCHIVO)
b, _ = limpiar(RUTA + ARCHIVO)
print('¿Reproducible?', a.equals(b))

# Guardar dataset limpio en español para la Semana 3
limpio.to_csv(RUTA + 'mi_conjunto_limpio.csv', sep=';', index=False)
print('Guardado exitoso:', limpio.shape)

Informe de ejecución: {'filas_iniciales': 1599, 'duplicados_eliminados': 240, 'atipicos_tratamiento': 'Conservados (fenómeno real)', 'filas_finales': 1359}
¿Reproducible? True
Guardado exitoso: (1359, 13)


In [49]:
# Comprobación: ejecuta la función DOS VECES y verifica que da lo mismo.
a, _ = limpiar(RUTA + ARCHIVO)
b, _ = limpiar(RUTA + ARCHIVO)
print('¿Reproducible?', a.equals(b))

¿Reproducible? True


In [50]:
# Guarda el conjunto limpio. Este archivo es el insumo de la semana 3.
limpio.to_csv(RUTA + 'mi_conjunto_limpio.csv', sep=';', index=False)
print('Guardado:', limpio.shape)

Guardado: (1359, 13)


---
## Parte 5 · Versionado

Sube el cuaderno y el conjunto limpio a tu repositorio. Un commit con mensaje descriptivo, no «cambios».

In [ ]:
# En una terminal:
# git add S2_Notebook_Laboratorio_CDD.ipynb mi_conjunto_limpio.csv
# git commit -m 'Semana 2: diagnostico de calidad y flujo de limpieza'
# git push


**Enlace a mi repositorio:** _…_

---

## Antes de cerrar el laboratorio

- [ ] Las cinco dimensiones diagnosticadas y registradas en la hoja «Resumen de calidad»
- [ ] Cada transformación aplicada tiene su fila en la bitácora, con justificación
- [ ] Me pronuncié sobre los atípicos: cuáles son error y cuáles fenómeno real
- [ ] Creé al menos un atributo derivado y expliqué para qué sirve
- [ ] La función `limpiar()` corre de principio a fin y entrega lo mismo dos veces
- [ ] El conjunto limpio está guardado y versionado
- [ ] Tengo anotadas las limitaciones que declararé en el informe

Lo que quede pendiente, al Foro de Consultas o a la sesión sincrónica.

_Ciencia de Datos · Advance UCSC · Semana 2_